# Geological Surface Accuracy





# 0.1 Load the workspace

Run the "LOAD WORKSPACE" cell only when opening the notebook (otherwise you overwrite your edits with the default workspace):

- It loads the working environment and the folder "working_files_folder" where you must place the GOCAD (.ts) file containing all surfaces of the 3D model and the shapefiles with section traces and well locations.


In [ ]:
# ### LOAD WORKSPACE

import os
import shutil

# Base path
base_path = '/content'
repo_path = os.path.join(base_path, 'GeoSurface_Accuracy')

# Function to fully clean the directory
def clean_repo_directory():
    try:
        # Remove the directory if it exists
        if os.path.exists(repo_path):
            shutil.rmtree(repo_path)
            print(f"Directory {repo_path} removed")
    except Exception as e:
        print(f"Error removing directory: {e}")

# Clean the directory
clean_repo_directory()

# Change to the base directory
os.chdir(base_path)

# Clone the repository
!git clone https://github.com/BaterHub/GeoSurface_Accuracy.git

# Change into the repository directory
%cd GeoSurface_Accuracy


# 0.2 Add files to folder "working_files_folder"

Drag the 3D model package files into "working_files_folder"

*You need to load:*
- horizons.ts (must contain all surface geometries)
- shapefiles of geological sections and seismic lines used to build the surface
- association files for surfaces/wells and surfaces/sections (surface_checkpoint_edges.csv) + the input selection file (surface_data_mapping.csv)


# 0.3 Run the script

- Go to the cell "RUN THE SCRIPT" and run with Ctrl + F10 or via "Runtime > Run cell and below".

- When the run ends, outputs are written inside "output_files_folder"


# 1. Import libraries and functions



In [ ]:
# ### RUN THE SCRIPT

## Import required libraries
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from pyproj import Proj, transform
from scipy.spatial import cKDTree
from scipy.interpolate import griddata
from sklearn.preprocessing import MinMaxScaler
import os
import re
from pathlib import Path

#############################################################################################

## Import functions
import importlib  # module to reload functions

# Reimport original modules
import files_utils

# Force reload of each module
importlib.reload(files_utils)

# Reimport functions from reloaded modules
from files_utils import *

#############################################################################################

# Folder paths
folder_name = "working_files_folder"
input_dir = os.path.abspath(folder_name)
output_dir = "output_results"


In [ ]:
# ### Working folder setup
working_dir = "working_files_folder"
output_dir = "output_results"
crs = 'EPSG:6708'
grid_spacing = 500  # grid node spacing (meters)


In [ ]:
# Main function and execution


def main(working_dir=working_dir, grid_spacing=grid_spacing):
    print("Starting geological data analysis...")

    if not os.path.exists(working_dir):
        print(f"The folder {working_dir} does not exist. Creating it...")
        os.makedirs(working_dir)
        print(f"Folder {working_dir} created. Place the GOCAD .ts file and shapefiles inside.")
        return None

    print(f"Files in {working_dir}:")
    for file in os.listdir(working_dir):
        print(f"  - {file}")

    ts_files = [f for f in os.listdir(working_dir) if f.endswith('.ts')]
    if not ts_files:
        print("No .ts file found.")
        return None
    ts_path = os.path.join(working_dir, ts_files[0])

    surfaces_data = read_gocad_ts_multi(ts_path)
    surface_names = list(surfaces_data.keys())
    if not surface_names:
        print("No surface found in the .ts file.")
        return None

    wells_all = read_wells_shapefile(working_dir)
    sections_all = read_sections_shapefile(working_dir)
    wells_depths = files_utils._load_depth_csv(working_dir, 'surface_checkpoint_depths_wells.csv', ['surface', 'checkpoint_id', 'z'])
    sections_depths = files_utils._load_depth_csv(working_dir, 'surface_checkpoint_depths_sections.csv', ['surface', 'checkpoint_id', 'z'])

    # Global extents for consistent axes (include vertices + wells + sections)
    all_vert_list = [v for v in surfaces_data.values() if v.get('vertices') is not None and len(v.get('vertices')) > 0]
    if not all_vert_list:
        print("No vertices found across surfaces.")
        return None
    all_xyz = np.vstack([v['vertices'] for v in all_vert_list])
    global_xmin, global_ymin = np.min(all_xyz[:, :2], axis=0)
    global_xmax, global_ymax = np.max(all_xyz[:, :2], axis=0)

    if wells_all is not None and not wells_all.empty:
        global_xmin = min(global_xmin, wells_all.geometry.x.min())
        global_xmax = max(global_xmax, wells_all.geometry.x.max())
        global_ymin = min(global_ymin, wells_all.geometry.y.min())
        global_ymax = max(global_ymax, wells_all.geometry.y.max())

    if sections_all is not None and not sections_all.empty:
        bounds = sections_all.geometry.bounds
        global_xmin = min(global_xmin, bounds.minx.min())
        global_xmax = max(global_xmax, bounds.maxx.max())
        global_ymin = min(global_ymin, bounds.miny.min())
        global_ymax = max(global_ymax, bounds.maxy.max())

    global_xlim = (global_xmin, global_xmax)
    global_ylim = (global_ymin, global_ymax)

    mapping_flags = {name: ensure_mapping_file(working_dir, name) for name in surface_names}
    edges_df = ensure_checkpoint_edges_file(working_dir, surface_names, wells_all, sections_all)

    results = {}
    all_vertices = []

    for sname, data in surfaces_data.items():
        print(f"\n--- Surface: {sname} ---")
        vertices = data.get('vertices')
        triangles = data.get('triangles')
        if vertices is None or len(vertices) == 0:
            print("No vertices for this surface, skipping.")
            continue
        all_vertices.append(vertices)

        flags = mapping_flags.get(sname, {'use_wells': True, 'use_sections': True, 'use_maps': False, 'use_vertical': True})
        wells_use = wells_all if flags.get('use_wells', True) else None
        sections_use = sections_all if flags.get('use_sections', True) else None
        maps_flag = flags.get('use_maps', False)

        wells_filt, sections_filt = filter_checkpoints_by_edges(edges_df, sname, wells_use, sections_use)

        has_wells = wells_filt is not None and not wells_filt.empty
        has_sections = sections_filt is not None and not sections_filt.empty
        print(f"  Wells used: {'yes' if has_wells else 'no'}")
        print(f"  Sections used: {'yes' if has_sections else 'no'}")
        print(f"  Maps flag: {'yes' if maps_flag else 'no (not implemented)'}")

        acc_outputs = generate_accuracy_outputs(vertices, wells_filt, sections_filt, output_dir,
                                                use_wells=has_wells, use_sections=has_sections,
                                                grid_spacing=grid_spacing, line_step=2000, surface_name=sname,
                                                xlim=global_xlim, ylim=global_ylim, crs_proj=crs)

        vert_outputs = None
        if flags.get('use_vertical', True):
            try:
                vert_outputs = generate_vertical_outputs(vertices, triangles, wells_filt, sections_filt,
                                                         acc_outputs.get('grid_points'),
                                                         acc_outputs.get('GX'), acc_outputs.get('GY'),
                                                         acc_outputs.get('mask'), output_dir, sname, idw_power=2,
                                                         well_depths_df=wells_depths, section_depths_df=sections_depths,
                                                         well_id_field='NOME_POZZO', section_id_field='NOME',
                                                         xlim=global_xlim, ylim=global_ylim, crs_proj=crs)
                if vert_outputs:
                    print(f"Vertical confidence calculated with {vert_outputs.get('samples', 0)} checkpoints.")
            except Exception as e:
                print(f"Error during vertical confidence computation: {e}")

        combined_outputs = None
        if vert_outputs is not None:
            combined_outputs = generate_combined_confidence(acc_outputs, vert_outputs, output_dir, sname, crs_proj=crs, alpha=0.5, mode="min")  # "geometric", "arithmetic", "min")

        try:
            fig = visualize_data(vertices, triangles, wells_filt, sections_filt, apply_smoothing=False,
                                 smoothing_iterations=3, smoothing_factor=0.2, crs='EPSG:6708',
                                 output_filename=f'model_dataset_{sname}.png',
                                 grid_points=acc_outputs.get('grid_points'), surface_name=sname, show_plot=False,
                                 xlim=global_xlim, ylim=global_ylim)
            print("Visualization completed successfully.")
        except Exception as e:
            print(f"Error during visualization: {e}")
            import traceback
            traceback.print_exc()

        results[sname] = {
            'vertices': vertices,
            'triangles': triangles,
            'wells': wells_filt,
            'sections': sections_filt,
            'grid_points': acc_outputs.get('grid_points'),
            'horizontal_weights': acc_outputs.get('weights'),
            'vertical_confidence': vert_outputs,
            'combined_confidence': combined_outputs
        }

    if all_vertices:
        try:
            all_vertices_arr = np.vstack(all_vertices)
            fig = visualize_data(all_vertices_arr, None, wells_all, sections_all, apply_smoothing=False,
                                 smoothing_iterations=0, smoothing_factor=0.0, crs='EPSG:6708',
                                 output_filename='model_dataset.png', grid_points=None,
                                 surface_name='model', show_plot=False,
                                 xlim=global_xlim, ylim=global_ylim)
            print("Combined visualization saved (model_dataset.png).")
        except Exception as e:
            print(f"Error during combined visualization: {e}")
            import traceback
            traceback.print_exc()

    print("Analysis completed.")
    return results


if __name__ == "__main__":
    data = main()